In [1]:
import pandas as pd
import numpy as np
import dai

def main(datasources: dict, start_date: str, end_date: str) -> pd.DataFrame:
    """
    优化版恐慌波动因子
    逻辑：
    1. 个股振幅 = (high - low) / close
    2. 个股20日振幅中位数（历史，T-1日及以前）
    3. 全市场振幅中位数及其5日加速度
    4. 因子 = (个股振幅 / 个股历史中位数) * (1 + 0.2 * 加速度)
    5. 取反（使因子正向，低波动选股）
    6. 行业 + 市值中性化（保留残差）
    7. 去极值（1%～99%）、平滑（5日均线）
    """
    bar1m = datasources["bar1m"]
    LOOKBACK = 120                # 增加缓冲，确保有足够历史计算20日/60日均线
    WINDOW = 20                   # 个股历史振幅窗口
    ACCEL_WINDOW = 5              # 加速度窗口

    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    query_start = (start_dt - pd.Timedelta(days=LOOKBACK)).strftime('%Y-%m-%d %H:%M:%S')

    # ---- 1. 聚合日线数据（带ORDER BY） ----
    sql_daily = f"""
    WITH daily_bar AS (
        SELECT
            date::DATE AS date,
            instrument,
            FIRST(open ORDER BY date)   AS open,
            MAX(high)                   AS high,
            MIN(low)                    AS low,
            LAST(close ORDER BY date)   AS close,
            SUM(volume)                 AS volume,
            SUM(amount)                 AS amount
        FROM {bar1m}
        WHERE close > 0 AND volume > 0
        GROUP BY date::DATE, instrument
    )
    SELECT *
    FROM daily_bar
    """
    df = dai.query(sql_daily, filters={'date': [query_start, end_date]}, compression=True).df()
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)

    for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['open', 'high', 'low', 'close'])

    # ---- 2. 对齐中证1000成分股（宽区间） ----
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [query_start, end_date]},
        compression=True
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    df['instrument'] = df['instrument'].astype(str)
    df = df.merge(stk_pool, on=['date', 'instrument'], how='inner')
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # ---- 3. 计算振幅 ----
    df['amp'] = (df['high'] - df['low']) / df['close'].replace(0, np.nan)
    df = df.dropna(subset=['amp'])

    # ---- 4. 个股20日振幅中位数（历史，shift(1)） ----
    df = df.sort_values(['instrument', 'date'])
    df['amp_median_20'] = df.groupby('instrument')['amp'].transform(
        lambda x: x.shift(1).rolling(WINDOW, min_periods=1).median()
    )

    # ---- 5. 市场恐慌烈度（全市场振幅中位数）及加速度 ----
    daily_market = df.groupby('date')['amp'].median().reset_index(name='market_median')
    daily_market['market_lag5'] = daily_market['market_median'].shift(ACCEL_WINDOW)
    daily_market['acceleration'] = daily_market['market_median'] / daily_market['market_lag5'] - 1
    daily_market['acceleration'] = daily_market['acceleration'].clip(-0.5, 0.5)  # 限制极端值

    # 合并到df
    df = df.merge(daily_market[['date', 'acceleration']], on='date', how='left')

    # ---- 6. 计算原始因子（个股相对自身异常波动 × 市场加速度调节） ----
    df['factor_raw'] = df['amp'] / df['amp_median_20'].replace(0, np.nan)
    df['factor_raw'] = df['factor_raw'].fillna(1.0)  # 缺失时中性
    # 加速度调节：加速度>0 时放大因子，<0时缩小
    df['factor'] = df['factor_raw'] * (1 + 0.2 * df['acceleration'].fillna(0))
    # 取反（使因子与未来收益正相关，因为原始高波动股票未来收益通常较低）
    df['factor'] = -df['factor']

    # ---- 7. 行业 + 市值中性化（利用exposure表） ----
    exposure = dai.query(
        """
        SELECT date, instrument, industry_level1_code, SIZE
        FROM bigalpha_2026_exposure
        """,
        filters={'date': [query_start, end_date]},
        compression=True
    ).df()
    exposure['date'] = pd.to_datetime(exposure['date'])
    exposure['instrument'] = exposure['instrument'].astype(str)
    exposure['industry_level1_code'] = exposure['industry_level1_code'].astype(str)

    # 合并暴露度
    df = df.merge(exposure[['date', 'instrument', 'industry_level1_code', 'SIZE']],
                  on=['date', 'instrument'], how='left')
    # 若某些股票缺失行业（极少），用代码前3位补全
    df['industry_level1_code'] = df['industry_level1_code'].fillna(df['instrument'].str[:3])
    df['SIZE'] = df['SIZE'].fillna(df.groupby('date')['SIZE'].transform('mean'))

    # 按日期分组进行中性化（回归残差）
    def neutralize(group):
        if len(group) < 5:
            group['factor_neu'] = group['factor']
            return group
        # 构造哑变量（行业）
        dummies = pd.get_dummies(group['industry_level1_code'], drop_first=True)
        X = dummies.copy()
        X['SIZE'] = group['SIZE']
        X = X.values
        # 添加常数项
        X = np.hstack([np.ones((X.shape[0], 1)), X])
        y = group['factor'].values
        # 最小二乘求解
        try:
            beta = np.linalg.lstsq(X, y, rcond=None)[0]
            residuals = y - X @ beta
            group['factor_neu'] = residuals
        except:
            group['factor_neu'] = group['factor']
        return group

    df = df.groupby('date', group_keys=False).apply(neutralize)

    # 因子替换为中性化后的值，并确保无inf
    df['factor'] = df['factor_neu'].replace([np.inf, -np.inf], np.nan)
    # 若某天全为NaN，则用原始因子填充（理论上不会发生）
    df['factor'] = df.groupby('date')['factor'].transform(
        lambda x: x.fillna(x.mean())
    )

    # ---- 8. 去极值（1%~99%）和平滑（5日均线） ----
    # 先按日期截面去极值
    def winsorize_series(s):
        q_low = s.quantile(0.01)
        q_high = s.quantile(0.99)
        return s.clip(q_low, q_high)

    df['factor'] = df.groupby('date')['factor'].transform(winsorize_series)

    # 个股5日平滑（仅用历史）
    df['factor'] = df.groupby('instrument')['factor'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    # ---- 9. 裁剪到评估区间 ----
    df = df[(df['date'] >= start_dt) & (df['date'] <= end_dt)]
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # ---- 10. 对齐评估区间成分股（确保全覆盖） ----
    stk_pool_eval = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
        compression=True
    ).df()
    stk_pool_eval['date'] = pd.to_datetime(stk_pool_eval['date'])
    stk_pool_eval['instrument'] = stk_pool_eval['instrument'].astype(str)

    final_df = pd.merge(
        stk_pool_eval,
        df[['date', 'instrument', 'factor']],
        how='left',
        on=['date', 'instrument']
    )

    # ---- 11. 缺失值填充（向前填充后填0） ----
    final_df['factor'] = final_df.groupby('instrument')['factor'].transform(
        lambda x: x.fillna(method='ffill')
    )
    final_df['factor'] = final_df['factor'].fillna(0.0)
    final_df['factor'] = final_df['factor'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # 再次检查，保证无缺失
    assert final_df['factor'].isnull().sum() == 0, "仍有缺失值！"

    # ---- 12. 输出 ----
    result = final_df[['date', 'instrument', 'factor']].sort_values(['date', 'instrument'])
    return result.reset_index(drop=True)


if __name__ == '__main__':
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算优化版恐慌波动因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    print(f"因子形状: {factor_data.shape}")
    print(factor_data.head())
    print(f"因子描述统计:\n{factor_data['factor'].describe()}")

    # 评估（可选）
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )

[2026-07-19 18:21:47] [info     ] 计算优化版恐慌波动因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
